# Lab 02 — One assistant, two SDKs: OpenAI function calling vs Claude tool use (solution)

> Reference solution: every TODO is filled in and every check passes.

**Level:** Beginner → Intermediate · **Duration:** ~60 minutes · **Covers:** H3 AI assistant with the OpenAI SDK: function calling + streaming (30 min) · H4 rebuild with the Claude SDK: tool use + streaming, compare API design (30 min)

Lab guide: [`../lab-02-one-assistant-two-sdks.md`](../lab-02-one-assistant-two-sdks.md) — read each part there first; this notebook holds the code.

You build the same IT operations assistant twice, once per SDK, on the same tools. Two functions are yours (marked `TODO` in Part 1); a check cell tests their contracts. Everything else is given: run it, compare, and break it on purpose.

In [ ]:
# (notebook) Setup: run this cell first. It loads the repo-root .env and creates both SDK clients.
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import anthropic
import openai
import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # walks up from this folder to the repo-root .env
pd.set_option("display.max_colwidth", 110)

# labs/module-02 holds data/ and templates/; the notebook runs from notebooks/ or solutions/.
LAB_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents) if p.name == "module-02" and (p / "data").is_dir())
DATA_DIR = LAB_DIR / "data"

# Model IDs come from .env, never from code (labs/setup/.env.example has the defaults).
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "").strip()
ANTHROPIC_MODEL = os.getenv("ANTHROPIC_MODEL", "").strip()

# Each SDK reads its key from the environment; a client exists only when key and model are set.
openai_client = openai.OpenAI() if os.getenv("OPENAI_API_KEY") and OPENAI_MODEL else None
claude_client = anthropic.Anthropic() if os.getenv("ANTHROPIC_API_KEY") and ANTHROPIC_MODEL else None
PROVIDERS = [p for p, c in (("openai", openai_client), ("anthropic", claude_client)) if c is not None]


def claude_text(message) -> str:
    """Join the text blocks of a Claude response; thinking blocks are skipped."""
    return "".join(block.text for block in message.content if block.type == "text")


def run_parallel(fn, jobs: list[tuple], max_workers: int = 8) -> list:
    """Call fn(*job) for every job on a small thread pool; results come back in job order."""
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        return list(pool.map(lambda job: fn(*job), jobs))


def check(name: str, fn) -> None:
    """Run a check function. A returned string means 'skipped'; a TODO is reported, not raised."""
    try:
        skipped = fn()
    except NotImplementedError as exc:
        print(f"⏳ {name}: TODO not implemented yet ({exc})")
        return
    print(f"⚠️  {name}: skipped ({skipped})" if skipped else f"✅ {name}")


print(f"Lab folder : {LAB_DIR}")
print(f"OpenAI     : {'✅ ' + OPENAI_MODEL if openai_client else '❌ OPENAI_API_KEY or OPENAI_MODEL missing'}")
print(f"Claude     : {'✅ ' + ANTHROPIC_MODEL if claude_client else '❌ ANTHROPIC_API_KEY or ANTHROPIC_MODEL missing'}")
if not PROVIDERS:
    print(
        "\n❌ No provider is configured. Copy labs/setup/.env.example to the repo-root .env, add your keys,\n"
        "   restart the kernel and run this cell again. Until then, cells that call a model are skipped."
    )

## Part 1 — Tools as contracts (5 min)

A tool is an RPC contract: a name, a description the model reads, a JSON Schema for the
arguments, and your function behind it. Define each tool once, then translate it into each
provider's format.

In [ ]:
# (notebook) The data behind the tools, and the tool functions: read-only, never raising.
import re
from collections import Counter
from dataclasses import dataclass, field
from typing import Any, Callable

INCIDENTS = {i["incident_id"]: i for i in json.loads((DATA_DIR / "incidents.json").read_text(encoding="utf-8"))}
KB = json.loads((DATA_DIR / "kb_articles.json").read_text(encoding="utf-8"))
SERVICES = json.loads((DATA_DIR / "service_status.json").read_text(encoding="utf-8"))
PRICES = json.loads((DATA_DIR / "model_prices.json").read_text(encoding="utf-8"))["models"]
WORD = re.compile(r"[a-z0-9]+(?:-[a-z0-9]+)*")


def lookup_incident(incident_id: str) -> dict:
    incident_id = incident_id.strip().upper()
    if not re.fullmatch(r"INC-\d{6}", incident_id):
        return {"error": "invalid_id", "incident_id": incident_id, "expected": "INC- followed by 6 digits"}
    return INCIDENTS.get(incident_id) or {"error": "not_found", "incident_id": incident_id}


def search_kb(query: str, max_results: int) -> list[dict]:
    terms = set(WORD.findall(query.lower()))
    scored = []
    for a in KB:
        score = (3 * len(terms & set(WORD.findall(" ".join(a["tags"]).lower())))
                 + 2 * len(terms & set(WORD.findall(a["title"].lower())))
                 + len(terms & set(WORD.findall(a["summary"].lower()))))
        if score:
            scored.append((score, a))
    scored.sort(key=lambda pair: (-pair[0], pair[1]["kb_id"]))
    return [{"kb_id": a["kb_id"], "title": a["title"], "summary": a["summary"], "escalation": a["escalation"]}
            for _, a in scored[: max(1, min(max_results, 5))]]


def get_service_status(service: str) -> list[dict] | dict:
    needle = service.strip().lower()
    hits = [s for s in SERVICES if needle and (needle in s["ci_id"].lower() or needle in s["service"].lower())]
    return hits or {"error": "not_found", "service": service}


print(json.dumps(lookup_incident("INC-104512"), indent=1)[:400], "…")
print([a["kb_id"] for a in search_kb("vpn rejects password", 3)])

In [ ]:
# (notebook) One neutral definition per tool. strict=True needs every property required and no extras.
@dataclass(frozen=True)
class ToolSpec:
    name: str
    description: str
    parameters: dict
    fn: Callable[..., Any]


def object_schema(**properties: dict) -> dict:
    return {"type": "object", "properties": properties, "required": list(properties), "additionalProperties": False}


TOOLS = [
    ToolSpec("lookup_incident",
             "Look up an existing incident by its ID. Returns status, priority, assignment group, CI, related change and updates.",
             object_schema(incident_id={"type": "string", "description": "INC- followed by 6 digits, e.g. INC-104233"}),
             lookup_incident),
    ToolSpec("search_kb",
             "Search the IT knowledge base by problem description. Returns matching articles with their escalation rule.",
             object_schema(query={"type": "string", "description": "Short problem description, e.g. 'vpn rejects new password'"},
                           max_results={"type": "integer", "description": "1 to 5; use 3 if unsure"}),
             search_kb),
    ToolSpec("get_service_status",
             "Current status of a service or configuration item: a CI ID such as 'vpn-gateway-eu-01' or a service name such as 'VPN'.",
             object_schema(service={"type": "string", "description": "CI ID or service name"}),
             get_service_status),
]
TOOL_BY_NAME = {t.name: t for t in TOOLS}


def to_openai_tool(spec: ToolSpec) -> dict:
    """Responses API function tool: flat, with `parameters`."""
    return {"type": "function", "name": spec.name, "description": spec.description,
            "parameters": spec.parameters, "strict": True}

In [ ]:
# (notebook) Your first TODO: the Claude format, and one dispatcher both loops will share.
def to_claude_tool(spec: ToolSpec) -> dict:
    """Messages API tool: `name`, `description`, the schema under `input_schema`, and `strict`."""
    return {"name": spec.name, "description": spec.description, "input_schema": spec.parameters, "strict": True}


def dispatch(name: str, arguments: str | dict) -> str:
    """Run tool `name` and return its result as a JSON string. Never raises.

    - `arguments` is a JSON string (OpenAI) or an already-parsed dict (Claude).
    - Unknown tool         -> {"error": "unknown_tool", "tool": name}
    - Arguments not JSON   -> {"error": "invalid_json", "detail": ...}
    - Wrong argument names -> {"error": "bad_arguments", "detail": ...}   (a TypeError from the call)
    - Any other exception  -> {"error": "tool_failed", "detail": ...}
    """
    spec = TOOL_BY_NAME.get(name)
    if spec is None:
        return json.dumps({"error": "unknown_tool", "tool": name})
    try:
        args = json.loads(arguments) if isinstance(arguments, str) else dict(arguments)
    except json.JSONDecodeError as exc:
        return json.dumps({"error": "invalid_json", "detail": exc.msg})
    try:
        result = spec.fn(**args)
    except TypeError as exc:
        return json.dumps({"error": "bad_arguments", "detail": str(exc)})
    except Exception as exc:
        return json.dumps({"error": "tool_failed", "detail": f"{exc.__class__.__name__}: {exc}"})
    return json.dumps(result, default=str)

In [ ]:
# (notebook) Check: both tool formats and the dispatcher meet their contracts.
def check_part1():
    claude_tool = to_claude_tool(TOOLS[0])
    assert set(claude_tool) == {"name", "description", "input_schema", "strict"}, claude_tool
    assert claude_tool["input_schema"]["additionalProperties"] is False
    assert json.loads(dispatch("lookup_incident", '{"incident_id": "INC-104512"}'))["priority"] == "P2"
    assert json.loads(dispatch("lookup_incident", {"incident_id": "INC-104512"}))["priority"] == "P2"
    assert json.loads(dispatch("reset_password", "{}"))["error"] == "unknown_tool"
    assert json.loads(dispatch("search_kb", "{not json"))["error"] == "invalid_json"
    assert json.loads(dispatch("search_kb", {"q": "vpn"}))["error"] == "bad_arguments"


check("Part 1 — tool contracts and dispatcher", check_part1)

OPENAI_TOOLS = [to_openai_tool(t) for t in TOOLS]
try:
    CLAUDE_TOOLS = [to_claude_tool(t) for t in TOOLS]
except NotImplementedError:
    CLAUDE_TOOLS = []
print(json.dumps(OPENAI_TOOLS[0], indent=1))

## Part 2 — The assistant on the OpenAI Responses API: function calling (15 min)

The loop is yours: send the request with tools, run every `function_call` the model returns,
send the results back as `function_call_output` items with the same `call_id`, and repeat until
the model answers in text. A `Trace` records every step for Part 5.

In [ ]:
# (notebook) A small trace recorder: one row per model call, tool call and tool result.
SYSTEM = (
    "You are the IT operations assistant for the Service Desk. Use the tools to look up incidents, "
    "knowledge-base articles and service status. Answer only from tool results; if the tools do not have "
    "the answer, say so. You cannot change anything: never claim that you reset, fixed, unlocked or updated something. "
    "Keep answers under 120 words."
)


@dataclass
class Trace:
    provider: str
    question: str
    steps: list[dict] = field(default_factory=list)
    started: float = field(default_factory=time.perf_counter)
    first_token_s: float | None = None
    answer: str = ""

    def add(self, kind: str, **data) -> None:
        self.steps.append({"t (s)": round(time.perf_counter() - self.started, 2), "kind": kind, **data})

    def first_token(self) -> None:
        if self.first_token_s is None:
            self.first_token_s = round(time.perf_counter() - self.started, 2)

    def totals(self) -> dict:
        calls = [s for s in self.steps if s["kind"] == "model call"]
        tokens_in, tokens_out = sum(s["in"] for s in calls), sum(s["out"] for s in calls)
        price = PRICES.get(MODELS[self.provider])
        cost = (tokens_in * price["input"] + tokens_out * price["output"]) / 1e6 if price else None
        return {"provider": self.provider, "model calls": len(calls),
                "tool calls": sum(s["kind"] == "tool call" for s in self.steps),
                "input tokens": tokens_in, "output tokens": tokens_out, "first token (s)": self.first_token_s,
                "wall (s)": self.steps[-1]["t (s)"] if self.steps else None,
                "cost ($)": round(cost, 5) if cost is not None else "—"}


MODELS = {"openai": OPENAI_MODEL, "anthropic": ANTHROPIC_MODEL}

In [ ]:
# (notebook) The OpenAI tool loop. With stream=True the same loop streams every model turn.
def run_openai(question: str, *, stream: bool = False, max_rounds: int = 6, tools: list | None = None) -> Trace:
    trace = Trace("openai", question)
    items: list = [{"role": "user", "content": question}]
    for _ in range(max_rounds):
        request = dict(model=OPENAI_MODEL, instructions=SYSTEM, input=items, tools=tools or OPENAI_TOOLS,
                       reasoning={"effort": "low"}, max_output_tokens=4_000)
        if stream:
            with openai_client.responses.stream(**request) as events:
                for event in events:
                    if event.type == "response.output_text.delta":
                        trace.first_token()
                        print(event.delta, end="", flush=True)
                response = events.get_final_response()
            print()
        else:
            response = openai_client.responses.create(**request)
        trace.add("model call", status=response.status, **{"in": response.usage.input_tokens, "out": response.usage.output_tokens})
        # Reasoning and call items go back in. The stream helper adds a client-side `parsed_arguments`
        # field to function calls; the API rejects it on input, so leave it out.
        items += [item.model_dump(exclude_none=True, exclude={"parsed_arguments"}) for item in response.output]
        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            trace.answer = response.output_text
            return trace
        for call in calls:
            trace.add("tool call", tool=call.name, args=call.arguments)
            output = dispatch(call.name, call.arguments)
            trace.add("tool result", tool=call.name, result=output[:120])
            items.append({"type": "function_call_output", "call_id": call.call_id, "output": output})
    trace.answer = f"[stopped after {max_rounds} rounds]"
    return trace


QUESTIONS = {
    "Q1": "What is the status of INC-104512, and is a change linked to it?",
    "Q2": "Users at EU-AMS-02 report VPN error 809 after changing their passwords. Is the EU VPN gateway healthy, and which KB article applies?",
    "Q3": "Hi! What can you help me with?",
}
traces = {}
if openai_client:
    for qid, question in QUESTIONS.items():
        traces["openai", qid] = run_openai(question)
        print(f"--- {qid}: {traces['openai', qid].answer}\n")
    display(pd.DataFrame(traces["openai", "Q2"].steps))

## Part 3 — Streaming responses (10 min)

Users judge an assistant by its time to first token. Streaming does not make the model faster;
it shows the answer while it is being written. Run Q2 again with `stream=True`.

In [ ]:
# (notebook) Stream Q2: tool rounds stream too, but only the final turn has text to show.
if openai_client:
    streamed = run_openai(QUESTIONS["Q2"], stream=True)
    traces["openai", "Q2 streamed"] = streamed
    print(f"\nfirst token after {streamed.first_token_s}s · whole answer after {streamed.steps[-1]['t (s)']}s")

    # What actually arrives on the wire: count the event types of one streamed turn.
    with openai_client.responses.stream(model=OPENAI_MODEL, instructions=SYSTEM, input=QUESTIONS["Q1"],
                                        tools=OPENAI_TOOLS, reasoning={"effort": "low"}, max_output_tokens=4_000) as events:
        kinds = Counter(event.type for event in events)
    display(pd.Series(kinds, name="events").to_frame())

## Part 4 — Rebuild it with the Claude Messages API: tool use (15 min)

Same tools, same system prompt, same dispatcher. The differences are in the envelope: `tool_use`
content blocks instead of `function_call` items, `stop_reason == "tool_use"`, and the results go
back as `tool_result` blocks inside a **user** message.

In [ ]:
# (notebook) The Claude tool loop, with optional streaming through the SDK's stream helper.
def run_claude(question: str, *, stream: bool = False, max_rounds: int = 6, tools: list | None = None) -> Trace:
    trace = Trace("anthropic", question)
    messages: list = [{"role": "user", "content": question}]
    for _ in range(max_rounds):
        request = dict(model=ANTHROPIC_MODEL, max_tokens=4_000, system=SYSTEM, tools=tools or CLAUDE_TOOLS,
                       output_config={"effort": "low"}, messages=messages)
        if stream:
            with claude_client.messages.stream(**request) as events:
                for event in events:
                    if event.type == "text":
                        trace.first_token()
                        print(event.text, end="", flush=True)
                message = events.get_final_message()
            print()
        else:
            message = claude_client.messages.create(**request)
        trace.add("model call", status=message.stop_reason, **{"in": message.usage.input_tokens, "out": message.usage.output_tokens})
        messages.append({"role": "assistant", "content": message.content})  # thinking + tool_use blocks go back in
        if message.stop_reason != "tool_use":
            trace.answer = claude_text(message)
            return trace
        results = []
        for block in (b for b in message.content if b.type == "tool_use"):
            trace.add("tool call", tool=block.name, args=json.dumps(block.input))
            output = dispatch(block.name, block.input)
            trace.add("tool result", tool=block.name, result=output[:120])
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": output,
                            "is_error": '"error"' in output[:12]})
        messages.append({"role": "user", "content": results})
    trace.answer = f"[stopped after {max_rounds} rounds]"
    return trace


if claude_client and CLAUDE_TOOLS:
    for qid, question in QUESTIONS.items():
        traces["anthropic", qid] = run_claude(question)
        print(f"--- {qid}: {traces['anthropic', qid].answer}\n")
    streamed = run_claude(QUESTIONS["Q2"], stream=True)
    traces["anthropic", "Q2 streamed"] = streamed
    print(f"\nfirst token after {streamed.first_token_s}s · whole answer after {streamed.steps[-1]['t (s)']}s")
    display(pd.DataFrame(traces["anthropic", "Q2"].steps))
elif claude_client:
    print("⏳ Finish to_claude_tool() in Part 1 first.")

In [ ]:
# (notebook) The SDK can run the loop for you: the tool runner (beta) builds schemas from type hints and docstrings.
from anthropic import beta_tool


@beta_tool
def lookup_incident_tool(incident_id: str) -> str:
    """Look up an existing incident by its ID.

    Args:
        incident_id: INC- followed by 6 digits, e.g. INC-104233.
    """
    return json.dumps(lookup_incident(incident_id))


if claude_client:
    runner = claude_client.beta.messages.tool_runner(
        model=ANTHROPIC_MODEL, max_tokens=4_000, system=SYSTEM, tools=[lookup_incident_tool],
        output_config={"effort": "low"}, messages=[{"role": "user", "content": QUESTIONS["Q1"]}],
    )
    final = runner.until_done()
    print(final.stop_reason, "·", "".join(b.text for b in final.content if b.type == "text"))

In [ ]:
# (notebook) Check: both loops reached a final answer and used tools where they should.
def check_part4():
    if not traces:
        return "no provider configured"
    for (provider, qid), trace in traces.items():
        assert not trace.answer.startswith("[stopped"), f"{provider} {qid} hit max_rounds"
        tools_used = {s["tool"] for s in trace.steps if s["kind"] == "tool call"}
        if qid == "Q1":
            assert "lookup_incident" in tools_used, f"{provider} Q1 should call lookup_incident"
        if qid == "Q2":
            assert tools_used >= {"get_service_status", "search_kb"} or len(tools_used) >= 2, f"{provider} Q2 tools: {tools_used}"


check("Part 4 — both tool loops", check_part4)

## Part 5 — Compare the two APIs, break them on purpose, and fill the trace sheet (15 min)

Put the runs side by side, then trigger three failures you will meet in production: a tool
that fails, a request no tool can serve, and a schema the API rejects.

In [ ]:
# (notebook) Side by side: same questions, same tools, two APIs.
if traces:
    display(pd.DataFrame([{"question": qid, **t.totals()} for (p, qid), t in traces.items()]).sort_values(["question", "provider"]))

In [ ]:
# (notebook) Deliberate failure A: the tool fails. B: no tool can do what the user asks.
FAILURES = {
    "A tool error": "What is the status of INC-999999?",
    "B missing tool": "Please reset my VPN password and unlock my account, I'm user EMP-20417.",
}
CLAIMS_ACTION = re.compile(r"\b(I('ve| have)?|has been|have been|was)\s+(now\s+)?(reset|unlocked|fixed|updated|changed)\b", re.IGNORECASE)
RUNNERS = {"openai": run_openai, "anthropic": run_claude}
failure_rows = []
for provider in PROVIDERS:
    if provider == "anthropic" and not CLAUDE_TOOLS:
        continue
    for label, question in FAILURES.items():
        t = RUNNERS[provider](question)
        errors = [s["result"] for s in t.steps if s["kind"] == "tool result" and '"error"' in s["result"]]
        failure_rows.append({"provider": provider, "case": label, "tool errors seen": len(errors),
                             "claims an action?": bool(CLAIMS_ACTION.search(t.answer)), "answer": t.answer[:160]})
pd.DataFrame(failure_rows)

In [ ]:
# (notebook) Deliberate failure C: a strict schema the API rejects (additionalProperties is missing).
loose = {"type": "object", "properties": {"incident_id": {"type": "string"}}, "required": ["incident_id"]}
if openai_client:
    try:
        openai_client.responses.create(model=OPENAI_MODEL, input="Status of INC-104512?", max_output_tokens=500,
                                       tools=[{"type": "function", "name": "lookup_incident", "description": "Look up an incident",
                                               "parameters": loose, "strict": True}])
        print("ℹ️  openai accepted the schema")
    except openai.BadRequestError as exc:
        print(f"❌ openai → 400: {exc.message}")
if claude_client:
    try:
        claude_client.messages.create(model=ANTHROPIC_MODEL, max_tokens=500, messages=[{"role": "user", "content": "Status of INC-104512?"}],
                                      tools=[{"name": "lookup_incident", "description": "Look up an incident",
                                              "input_schema": loose, "strict": True}])
        print("ℹ️  anthropic accepted the schema")
    except anthropic.BadRequestError as exc:
        print(f"❌ anthropic → 400: {exc.body['error']['message'] if isinstance(exc.body, dict) else exc}")

In [ ]:
# (notebook) Fill the agent trace sheet for Q2 on each provider (templates/agent-trace-sheet.md -> outputs/).
from datetime import date

OUTPUT_DIR = LAB_DIR / "outputs"
TEMPLATE = (LAB_DIR / "templates" / "agent-trace-sheet.md").read_text(encoding="utf-8")


def markdown_table(rows: list[dict]) -> str:
    columns = list(dict.fromkeys(k for r in rows for k in r))
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join("---" for _ in columns) + " |"]
    for r in rows:
        lines.append("| " + " | ".join(str(r.get(c, "")).replace("|", "\\|").replace("\n", " ") for c in columns) + " |")
    return "\n".join(lines)


def export_trace_sheet(trace: Trace, question_id: str) -> Path:
    t = trace.totals()
    values = {
        "provider": trace.provider, "model": MODELS[trace.provider], "question_id": question_id,
        "api": "Responses API" if trace.provider == "openai" else "Messages API",
        "date": date.today().isoformat(), "question": trace.question, "steps_table": markdown_table(trace.steps),
        "totals_row": "| " + " | ".join(str(t[k]) for k in ("model calls", "tool calls", "input tokens", "output tokens",
                                                               "first token (s)", "wall (s)", "cost ($)")) + " |",
        "final_answer": trace.answer, "notes": "",
    }
    sheet = TEMPLATE
    for key, value in values.items():
        sheet = sheet.replace("{{" + key + "}}", str(value))
    sheet = re.sub(r"\{\{review_\w+\}\}", "", sheet)  # the review section is yours to fill in by hand
    OUTPUT_DIR.mkdir(exist_ok=True)
    path = OUTPUT_DIR / f"agent-trace-{trace.provider}-{question_id}.md"
    path.write_text(sheet, encoding="utf-8")
    return path


sheets = [export_trace_sheet(traces[p, "Q2"], "Q2") for p in PROVIDERS if (p, "Q2") in traces]
for path in sheets:
    print(f"Wrote {path.relative_to(LAB_DIR)}: open it and fill in section 4 (Review).")

In [ ]:
# (notebook) Check: the failures were handled without crashing, and the trace sheets are complete.
def check_part5():
    if not traces:
        return "no provider configured"
    assert failure_rows, "run the deliberate failures"
    assert all(r["tool errors seen"] >= 1 for r in failure_rows if r["case"] == "A tool error"), "the model should have seen the not_found error"
    for r in failure_rows:
        if r["claims an action?"]:
            print(f"⚠️  {r['provider']} claimed an action it cannot take: that is a finding for your trace sheet.")
    assert sheets and all("{{" not in p.read_text(encoding="utf-8") for p in sheets), "every placeholder must be filled"


check("Part 5 — comparison, failures and trace sheet", check_part5)

📝 **Write-up (3–5 sentences).** Where do the two APIs differ in ways that would change your code
(state, message shapes, errors, streaming events)? Which provider recovered better from the tool
error, and what does your trace sheet say you should change first?

> *Your answer here.*

---

**Done.** Answer the checkpoint questions in the lab guide, then go on to
[Lab 03 — Designing agents with Skills and OpenSpec](../lab-03-skills-and-openspec.md).